In [1]:
from pathlib import Path
import pyvista as pv
import numpy as np
import json

from phd_helpers.paths import transform_mesh, get_relative_transform_new_basis, get_subject_stl_path, get_bone_transforms, pose2idCMC, get_bone_inertia, get_boundary
from phd_helpers.AbaqusPreprocessing import position_mc1_tpm

In [132]:
mesh_path = Path('../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest/meshes')

sub_S = '50000R'
sub_M = '50017L'
sub_L = '50034R'

In [271]:
sub = sub_L
pose = 'pinch_load'
stl_path = get_subject_stl_path(sub[:-1], sub[-1])

# load meshes
tpm_mesh = pv.read(mesh_path / f'{sub}/tpm-mc1/3Dmesh/mesh-35T.vtu')
mc1_mesh = pv.read(mesh_path / f'{sub}/mc1-tpm/3Dmesh/mesh-35T.vtu')
guide_mesh = pv.read('../../../3Dprints/CADmodels/instronJigs/solidWorks/Guide.ply')
adaptor_mesh = pv.read('../../../3Dprints/CADmodels/instronJigs/solidWorks/Adaptor.ply')

# transform to mc1 axes
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
mc1_mesh = transform_mesh(mc1_mesh, mc1_axes, mc1_centroid, inverse=True)

# put tpm in pose
R, t = get_relative_transform_new_basis(get_bone_transforms(pose2idCMC(pose), stl_path), 'tpm', 'mc1', mc1_centroid, mc1_axes)
tpm_mesh = transform_mesh(tpm_mesh, R, t)

position_mc1_tpm(mc1_mesh, tpm_mesh, target_dist=1)

Distance between cartilage surfaces (x->): 1.0000
No interference:  True


In [192]:
pl = pv.Plotter()
pl.add_mesh(tpm_mesh, color='white')
pl.add_mesh(mc1_mesh, color='white', opacity=0.5)
pl.add_mesh(guide_mesh, color='white')
pl.add_mesh(adaptor_mesh, color='white')
pl.add_points(np.zeros(3), point_size=15)
pl.show()

Context leak detected, CoreAnalytics returned false


Widget(value='<iframe src="http://localhost:50055/index.html?ui=P_0x40fe81820_99&reconnect=auto" class="pyvist…

#### Find distance of guide ledge from origin (x-axis)
 - This corresponds to distance of guide channel from top of cylindrical jig
 - This plus 10 mm (mc1_protrusion) gives distance of peak of mc1 cartilage from guide ledge

In [195]:
guide_X = guide_mesh.points[:, 0]
guide_Y = guide_mesh.points[:, 1]
guide_Z = guide_mesh.points[:, 2]
np.unique(guide_X[(guide_X < 0) & (guide_Z > 0)])

pyvista_ndarray([-7.5, -2.5], dtype=float32)

Therefore, the distance between the upper guide legs and the peak of the mc1 cartilage is 12.5 mm (+0.1 mm ? but don't include this)

#### Find distance of skinny ledge from origin (z-axis)

In [72]:
adaptor_mesh_X = adaptor_mesh.points[:, 0]
adaptor_mesh_Z = adaptor_mesh.points[:, 2]

adaptor_mesh_Z[(adaptor_mesh_X > -0.1) & (adaptor_mesh_Z > 0)].max()

np.float32(10.0)

In [73]:
# distance of back wall from origin
guide_Z.min()

np.float32(-10.0)

In [74]:
# thickness of guide 
np.unique(guide_Y[(guide_X < 0) & (guide_Z > 0)])
# 3 mm

pyvista_ndarray([-12.,  -9.,   9.,  12.], dtype=float32)

#### Measure tolerance

In [ ]:
# guide y vals
np.unique(guide_Y[(guide_X < 0) & (guide_Z > 0)])
# 9

pyvista_ndarray([-12.,  -9.,   9.,  12.], dtype=float32)

In [ ]:
point = np.array([-5., 9., 0.])
cell = adaptor_mesh.find_closest_cell(point)
faces = adaptor_mesh.faces.reshape(-1, 4)[:, 1:]
adaptor_mesh.points[:, 1][faces[cell]]
# 8.9

pyvista_ndarray([8.9, 8.9, 8.9], dtype=float32)

Therefore tolerance = 0.1 mm

In [ ]:
pl = pv.Plotter()
pl.add_mesh(adaptor_mesh)
pl.add_points(point, point_size=10, color='magenta')
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:50055/index.html?ui=P_0x4f1030890_103&reconnect=auto" class="pyvis…

## Split and position guide
 - split into legs/neck and arms
 - position correctly along x and on skinny side

In [272]:
offset_x = 12.5 # distance of upper surface of guide legs from peak of mc1 cartilage
offset_z = 10.0 # distance in z of skinny ledge from origin
# position top of legs of guide at origin
guide_mesh.points[:, 0] += 2.5
guide_mesh_mask = (guide_mesh.points[:, 0][guide_mesh.faces.reshape(-1, 4)[:, 1:]] > 0.1).all(axis=1) # mask of neck/arms
# reflect it
guide_mesh.points[:, 0] *= -1
# position top ledge of guide legs 12.5 mm from cartilage peak
cart_peak = min(mc1_mesh.points[:, 0])
guide_offset = cart_peak + offset_x
guide_mesh.points[:, 0] += guide_offset

# position inner wall at the origin
guide_mesh.points[:, 2] += 7 # distance of inner wall (3 mm thick) from origin
# reflect it
guide_mesh.points[:, 2] *= -1
# position at skinny side
guide_mesh.points[:, 2] += 10 #+ 1.5 + 2

# split
guide_arms = guide_mesh.extract_cells(guide_mesh_mask)
guide_legs = guide_mesh.extract_cells(guide_mesh_mask, invert=True)


# position adaptor to verify
adaptor_mesh.points[:, 0] *= -1
adaptor_mesh.points[:, 0] += cart_peak + 10

In [148]:
pl = pv.Plotter()
pl.add_mesh(mc1_mesh, color='white')
#pl.add_mesh(guide_mesh, color='gray')
pl.add_mesh(guide_arms, color='magenta')
pl.add_mesh(guide_legs, color='cyan')
pl.add_mesh(adaptor_mesh, color='gray')
pl.show()

Widget(value='<iframe src="http://localhost:50055/index.html?ui=P_0x35be2a540_71&reconnect=auto" class="pyvist…

# Clean code

In [17]:
def pose_bones(sub, pose, target_dist=1):
    stl_path = get_subject_stl_path(sub[:-1], sub[-1])

    # load meshes
    tpm_mesh = pv.read(mesh_path / f'{sub}/tpm-mc1/3Dmesh/mesh-35T.vtu')
    mc1_mesh = pv.read(mesh_path / f'{sub}/mc1-tpm/3Dmesh/mesh-35T.vtu')

    # transform to mc1 axes
    mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
    tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
    mc1_mesh = transform_mesh(mc1_mesh, mc1_axes, mc1_centroid, inverse=True)

    # put tpm in pose
    R, t = get_relative_transform_new_basis(get_bone_transforms(pose2idCMC(pose), stl_path), 'tpm', 'mc1', mc1_centroid, mc1_axes)
    tpm_mesh = transform_mesh(tpm_mesh, R, t)
    position_mc1_tpm(mc1_mesh, tpm_mesh, target_dist=target_dist)

    return tpm_mesh, mc1_mesh


def position_and_split(guide_mesh, mc1_mesh):
    guide_upper_legs = 2.5 # distance of guide upper legs surface from origin
    cart_legs_offset = 12.5 # distance of upper surface of guide legs from peak of mc1 cartilage
    guide_back_wall = 10.0 # distance of guide back wall from origin
    guide_thickness = 3.0 
    skinny_offset = 10 # distance of skinny side from origin

    # position top of legs of guide at origin
    guide_mesh.points[:, 0] += guide_upper_legs
    guide_arms_mask = (guide_mesh.points[:, 0][guide_mesh.faces.reshape(-1, 4)[:, 1:]] > 0.1).all(axis=1)
    guide_legs_mask = (guide_mesh.points[:, 0][guide_mesh.faces.reshape(-1, 4)[:, 1:]] < 0.1).all(axis=1)
    guide_neck_mask = ~(guide_arms_mask | guide_legs_mask) 
    # reflect it
    guide_mesh.points[:, 0] *= -1
    # position top ledge of guide legs 12.5 mm from cartilage peak
    cart_peak = min(mc1_mesh.points[:, 0])
    guide_offset = cart_peak + cart_legs_offset
    guide_mesh.points[:, 0] += guide_offset

    # position inner wall at the origin
    guide_mesh.points[:, 2] += guide_back_wall - guide_thickness # distance of inner wall (3 mm thick) from origin
    # reflect it
    guide_mesh.points[:, 2] *= -1
    # position at skinny side
    guide_mesh.points[:, 2] += skinny_offset

    # split
    guide_arms = guide_mesh.extract_cells(guide_arms_mask)
    guide_legs = guide_mesh.extract_cells(guide_legs_mask)
    guide_neck = guide_mesh.extract_cells(guide_neck_mask)
    return guide_arms, guide_legs, guide_neck

def adjust_arms(guide_arms, t=(0, 0, 0), R=(0, 0, 0), lower_adjust=0, upper_adjust=0):
    mesh = guide_arms.copy()

    # alter thickness
    eta = 0.1
    X = mesh.points[:, 0]
    low_x, high_x = X.max(), X.min()
    lower_mask = (X > low_x-eta) & (X < low_x+eta)
    upper_mask = (X > high_x-eta) & (X < high_x+eta)
    mesh.points[:, 0][lower_mask] += lower_adjust
    mesh.points[:, 0][upper_mask] -= upper_adjust

    # translate and rotate
    mesh.translate(t, inplace=True)
    cor = mesh.points.mean(axis=0)+np.array([0, 0, -2])
    mesh.rotate_x(R[0], point=cor, inplace=True)
    mesh.rotate_y(R[1], point=cor, inplace=True)
    mesh.rotate_z(R[2], point=cor, inplace=True)
    return mesh

In [27]:
mesh_path = Path('../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest/meshes')

# load jigs
guide_mesh = pv.read('../../../3Dprints/CADmodels/instronJigs/solidWorks/Guide.ply')
adaptor_mesh = pv.read('../../../3Dprints/CADmodels/instronJigs/solidWorks/Adaptor.ply')

# sub list
sub_S = '50000R'
sub_M = '50017L'
sub_L = '50034R'
poses = ['pinch_load', 'flexion', 'extension', 'abduction', 'adduction']

# params
sub = sub_M
pose = poses[0]
target_dist = 2

# load bones, position everything
tpm_mesh, mc1_mesh = pose_bones(sub, pose, target_dist=target_dist)
guide_arms, guide_legs, guide_neck = position_and_split(guide_mesh, mc1_mesh)

Distance between cartilage surfaces (x->): 2.0000
No interference:  True


In [28]:
z_adjust = 1.5 #+2 # 1.5 further back makes mc1 cart more central on sensor (added 2 mm spacer for big guy)
t = (0, 3.5, z_adjust) 
R = (0, 0, -15) # degrees rotation around each axis, cor at centre of guide after translation
upper_adjust = -2 # -ive skinnier, starts 8 mm thick
lower_adjust = 0 # -ive skinnier, starts 8 mm thick
new_arms = adjust_arms(guide_arms, t, R, lower_adjust, upper_adjust)

pl = pv.Plotter()
pl.add_mesh(mc1_mesh, color='white')
pl.add_mesh(tpm_mesh, color='white')
pl.add_mesh(guide_arms, color='black', style='wireframe', opacity=0.5)
pl.add_mesh(new_arms, color='yellow')
#pl.add_mesh(guide_legs, color='cyan')
pl.add_axes()
pl.camera_position = pv.CameraPosition(position=(-20.069779834558602, 5.016343429641955, -67.09762965991456),
               focal_point=(-16.59163822760403, 0.28785345346029323, 3.876228882402942),
               viewup=(-0.9988056184642292, -0.0018097566888099373, 0.048826850246726186))
pl.show()

Widget(value='<iframe src="http://localhost:55202/index.html?ui=P_0x347cafc50_12&reconnect=auto" class="pyvist…

#save per sub/pose params
params = {
    'sub': sub,
    'pose': pose,
    't': list(t),
    'R': list(R),
}

save_file = 'params.jsonl'
with open(save_file, 'a') as f:
    json.dump(params, f)
    f.write('\n')

# Build new guides

In [30]:
guide_mesh = pv.read('../../../3Dprints/CADmodels/instronJigs/solidWorks/Guide.ply')
guide_arms, guide_legs, guide_neck = position_and_split(guide_mesh, mc1_mesh)

#### Fill holes
- for all 3 sections
- check normals

In [31]:
# arms
guide_arms['point_id'] = np.arange(guide_arms.n_points)
arms_bound = get_boundary(guide_arms)
new_arms_faces = np.r_[guide_arms.cells, 3, arms_bound['point_id'][:-1], 3, arms_bound['point_id'][1:]]
guide_arms_filled = pv.PolyData(guide_arms.points, new_arms_faces)
guide_arms_filled.compute_normals(consistent_normals=True, inplace=True)
guide_arms_filled.is_manifold

True

In [32]:
# legs
guide_legs['point_id'] = np.arange(guide_legs.n_points)
legs_bound = get_boundary(guide_legs)
new_legs_faces = np.r_[guide_legs.cells, 3, legs_bound['point_id'][:-1], 3, legs_bound['point_id'][[2, 0, 3]]]
guide_legs_filled = pv.PolyData(guide_legs.points, new_legs_faces)
guide_legs_filled.compute_normals(consistent_normals=True, inplace=True)
guide_legs_filled.is_manifold

True

In [33]:
# neck
guide_neck['point_id'] = np.arange(guide_neck.n_points)
neck_bound = get_boundary(guide_neck)
hole1_ids = neck_bound['point_id'][[0, 1, 4, 6]]
hole2_ids = neck_bound['point_id'][[2, 3, 5, 7]]
new_neck_faces = np.r_[
            guide_neck.cells, 
            3, hole1_ids[:-1], 3, hole1_ids[1:], 
            3, hole2_ids[:-1], 3, hole2_ids[1:]
        ]
guide_neck_filled = pv.PolyData(guide_neck.points, new_neck_faces)
guide_neck_filled.compute_normals(consistent_normals=True, inplace=True)
guide_neck_filled.is_manifold

True

#### Reduce arms thickness and step back
 - reduce thickness by 2 mm
    - true for all guides
 - step arms back by 1.5 mm

In [34]:
guide_arms_filled_morphed = adjust_arms(guide_arms_filled, t=(0, 0, 1.5), upper_adjust=-2)

#### Extend legs and tolerance on inner legs
 - extend by 3.5 mm
 - current tolerance is 0.1 mm
    - thicken by 0.05 mm

In [35]:
legs_Z = guide_legs_filled.points[:, 2]
legs_Y = guide_legs_filled.points[:, 1]

extend_mask = legs_Z < 0
tol_mask_pos = (legs_Y > 8.5) & (legs_Y < 9.5)
tol_mask_neg = (legs_Y < -8.5) & (legs_Y > -9.5)
print(sum(extend_mask), sum(tol_mask_pos), sum(tol_mask_neg))

guide_legs_filled_morphed = guide_legs_filled.copy()
guide_legs_filled_morphed.points[:, 2][extend_mask] -= 3.5
guide_legs_filled_morphed.points[:, 1][tol_mask_pos] -= 0.05
guide_legs_filled_morphed.points[:, 1][tol_mask_neg] += 0.05

8 4 4


In [113]:
pl = pv.Plotter()
pl.add_mesh(guide_legs_filled, color='black', style='wireframe', opacity=0.5, line_width=3)
pl.add_mesh(guide_legs_filled_morphed)
pl.add_points(np.zeros(3), point_size=20)
pl.add_points(guide_legs_filled.points[extend_mask], color='magenta', point_size=30)
pl.add_points(guide_legs_filled.points[tol_mask_pos], color='cyan', point_size=20)
pl.add_points(guide_legs_filled.points[tol_mask_neg], color='blue', point_size=20)
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:52118/index.html?ui=P_0x357fbd9d0_50&reconnect=auto" class="pyvist…

#### Extend neck and step back
- make upward extension intersection work for all if possible (couple mm ?)
    - sub_L max rotation and others max rotation are the two opposite extremes
- extend down full leg thickness (5 mm)
- step back (1.5 mm)

In [36]:
neck_X = guide_neck_filled.points[:, 0]

eta = 0.1
neck_mask_down = neck_X > neck_X.max()-eta
neck_mask_up = neck_X < neck_X.min()+eta

print(sum(neck_mask_up), sum(neck_mask_down))

guide_neck_filled_morphed = guide_neck_filled.copy()
guide_neck_filled_morphed.points[:, 0][neck_mask_down] += 5 # extend down (flush with legs)
guide_neck_filled_morphed.points[:, 0][neck_mask_up] -= 3   # extend up (fit with all arms)
guide_neck_filled_morphed.points[:, 2] += 1.5 # step back

4 4


In [146]:
arms1 = adjust_arms(guide_arms, t=[0, 3.5, 1.5], R=[0, 0, -20], upper_adjust=-2)
arms2 = adjust_arms(guide_arms, t=[0, -2.5, 1.5], R=[0, 0, 15], upper_adjust=-2)
arms3 = adjust_arms(guide_arms, t=[1, -1.0, 1.5], R=[0, 0, 15], upper_adjust=-2)
arms4 = adjust_arms(guide_arms, t=[1, 2.5, 1.5], R=[0, 0, -10], upper_adjust=-2)

pl = pv.Plotter()
pl.add_mesh(guide_neck_filled, color='black', style='wireframe', opacity=0.5, line_width=3)
pl.add_mesh(guide_neck_filled_morphed)
pl.add_mesh(guide_legs_filled_morphed, color='gray', opacity=0.2)
pl.add_mesh(guide_arms_filled_morphed, color='gray', opacity=0.2)
pl.add_mesh(arms1, color='gray', opacity=0.2)
pl.add_mesh(arms2, color='gray', opacity=0.2)
pl.add_mesh(arms3, color='gray', opacity=0.2)
pl.add_mesh(arms4, color='gray', opacity=0.2)
pl.add_points(np.zeros(3), point_size=20)
pl.add_points(guide_neck_filled.points[neck_mask_up], color='cyan', point_size=20)
pl.add_points(guide_neck_filled.points[neck_mask_down], color='blue', point_size=20)
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:52118/index.html?ui=P_0x37cde6ba0_70&reconnect=auto" class="pyvist…

#### Final parts

In [37]:
pl = pv.Plotter()
pl.add_mesh(guide_neck_filled_morphed, color='yellow', style='wireframe', opacity=1, line_width=5)
pl.add_mesh(guide_arms_filled_morphed, color='magenta', style='wireframe', opacity=1, line_width=5)
pl.add_mesh(guide_legs_filled_morphed, color='cyan', style='wireframe', opacity=1, line_width=5)
pl.show()

Widget(value='<iframe src="http://localhost:55202/index.html?ui=P_0x360149fd0_13&reconnect=auto" class="pyvist…

## Boolean union

In [38]:
import trimesh
import pymeshfix
from trimesh.interfaces import blender
from phd_helpers.CartilageGeneration import get_trimesh, get_pvmesh

In [225]:
# convert to trimesh
arms_tmesh = get_trimesh(guide_arms_filled_morphed)
legs_tmesh = get_trimesh(guide_legs_filled_morphed)
neck_tmesh = get_trimesh(guide_neck_filled_morphed)
# boolean union
union_tmesh = blender.boolean([arms_tmesh, legs_tmesh, neck_tmesh], operation='union')
# remove 0 area triangles
mfix = pymeshfix.MeshFix(union_tmesh.vertices, union_tmesh.faces)
mfix.repair( joincomp=False, remove_smallest_components=False)
union_tmesh_fixed = trimesh.Trimesh( vertices=mfix.points, faces=mfix.faces, process=False)
# convert back to pyvista
union = get_pvmesh(union_tmesh_fixed)
# check all good
print(union_tmesh_fixed.is_watertight, union_tmesh_fixed.is_winding_consistent, union_tmesh_fixed.is_volume)
print(union.is_manifold, union.is_all_triangles)
print((union.compute_cell_sizes()['Area'] > 0).all())

True True True
True True
True


In [39]:
def build_guide(guide_arms, guide_legs, guide_neck):
    # convert to trimesh
    arms_tmesh = get_trimesh(guide_arms)
    legs_tmesh = get_trimesh(guide_legs)
    neck_tmesh = get_trimesh(guide_neck)
    # boolean union
    union_tmesh = blender.boolean([arms_tmesh, legs_tmesh, neck_tmesh], operation='union')
    # remove 0 area triangles
    mfix = pymeshfix.MeshFix(union_tmesh.vertices, union_tmesh.faces)
    mfix.repair( joincomp=False, remove_smallest_components=False)
    union_tmesh_fixed = trimesh.Trimesh( vertices=mfix.points, faces=mfix.faces, process=False)
    # convert back to pyvista
    union = get_pvmesh(union_tmesh_fixed)
    # check all good
    tmesh_check = union_tmesh_fixed.is_watertight & union_tmesh_fixed.is_winding_consistent & union_tmesh_fixed.is_volume
    pv_check = union.is_manifold & union.is_all_triangles
    area_check = (union.compute_cell_sizes()['Area'] > 0).all()
    if not (tmesh_check & pv_check & area_check): 
        raise AssertionError('Failed checks')
    return union

In [261]:
union = build_guide(guide_arms_filled_morphed, guide_legs_filled_morphed, guide_neck_filled_morphed)

In [262]:
pl = pv.Plotter()
pl.add_mesh(union, show_edges=True, line_width=5)
pl.show()

Widget(value='<iframe src="http://localhost:52118/index.html?ui=P_0x3ea2ed6d0_84&reconnect=auto" class="pyvist…

### Build them

#### Read in the params
 - set all z-translations to 0 
    - taken care of already or by spacer

In [40]:
params = []
with open("params.jsonl", "r") as f:
    for line in f:
        param = json.loads(line)
        param['t'][-1] = 0
        params.append(param)

unique_guides = set((tuple(line['t']), tuple(line['R']), line['guide_id']) for line in params)
guide_list = [{'t': x[0], 'R': x[1], 'guide_id': x[2]} for x in unique_guides]
guide_list

[{'t': (1, 2.5, 0), 'R': (0, 0, -10), 'guide_id': 5},
 {'t': (0, 0, 0), 'R': (0, 0, 0), 'guide_id': 0},
 {'t': (0, 2, 0), 'R': (0, 0, 0), 'guide_id': 2},
 {'t': (1, -1, 0), 'R': (0, 0, 15), 'guide_id': 6},
 {'t': (0, 3.5, 0), 'R': (0, 0, -15), 'guide_id': 1},
 {'t': (0, -2.5, 0), 'R': (0, 0, 15), 'guide_id': 3},
 {'t': (1, 0.5, 0), 'R': (0, 0, -10), 'guide_id': 4}]

#### Build and save

In [41]:
for param in guide_list:
    arms = adjust_arms(guide_arms_filled_morphed, t=param['t'], R=param['R'])
    union = build_guide(arms, guide_legs_filled_morphed, guide_neck_filled_morphed)
    union.save(f'guide_meshes/guide_{param['guide_id']}.stl')

In [ ]:
guide1 = pv.read('guide_meshes/guide_1.stl')

pl = pv.Plotter()
pl.add_mesh(guide1, color='gray')
pl.add_mesh(adaptor_mesh)
pl.show()

Widget(value='<iframe src="http://localhost:52118/index.html?ui=P_0x3ef0c1400_93&reconnect=auto" class="pyvist…

#### 2 mm spacer 
 - give plenty space between legs
 - legs are 9 apart on either side, so gap of 18

In [13]:
spacer = pv.Box(bounds=(0.0, 6.0, -6.0, 6.0, 0.0, 2.0))
spacer.save('spacer2mm.stl')

In [12]:
guide1 = pv.read('guide_meshes/guide_1.stl')

pl = pv.Plotter()
pl.add_mesh(guide1, color='gray')
pl.add_points(np.zeros(3), point_size=20)
pl.add_mesh(spacer)
pl.add_axes()
pl.show()

Widget(value='<iframe src="http://localhost:55202/index.html?ui=P_0x33c7e3650_6&reconnect=auto" class="pyvista…

In [ ]:

maybe don't project the sensor at all
 - we know angle and y and z
 - set x at best fit to cartilage surface
 - find sensel corners based on closest point 
 - use geodesic lines to connect corners and average elements within each 
OR 
find closest point in x between cart surfaces
 - use that and known y/z/orientation to map the sensor onto the surface 
 - could refine sensor mesh and just piecwise linear it from the sensor mesh element at initial contact
    - piecewise linear mesh can be much finer than sensor, break each sensel down into smaller squares.
        - each of those squares inherits that pressure of its most overlapping element
        - can do mesh refinement to determine how refined sensor mesh needs to be 
    - also do error based on how much sensor may have slipped while being pressed onto surface
I LIKE THIS ONE